In [5]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

from mpl_toolkits.axes_grid1 import make_axes_locatable


merge_iter_root = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/merge_resolution/'
starting_solution_root = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/scaled_w_merge/'
ending_solution_root = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/post_merge_resolution_merges'
all_metrics_info = pd.read_csv(merge_iter_root + 'all_iterations_metrics.csv')
all_merge_info = pd.read_csv(merge_iter_root + 'all_iterations_merge_info.csv')
unique_merges_inspected = pd.read_csv(merge_iter_root + 'num_unique_merges.csv')

In [ ]:
# compute proportion of remaining errors at each iteration
for column in ['fn_nodes', 'fp_edges', 'fn_edges']:
    new_col_name = f'{column}_prop'
    starting_col = all_metrics_info[all_metrics_info.iteration == 0][['ds_name', column]]
    starting_col = dict(zip(starting_col['ds_name'], starting_col[column]))
    all_metrics_info[new_col_name] = all_metrics_info.apply(lambda row: row[column] / starting_col[row['ds_name']], axis=1)

In [ ]:
# compute the proportion of edge errors remaining at each iteration
all_metrics_info['all_edge_errors'] = all_metrics_info['fp_edges'] + all_metrics_info['fn_edges']
starting_all_edge_error = all_metrics_info[all_metrics_info.iteration == 0][['ds_name', 'all_edge_errors']]
starting_all_edge_error = dict(zip(starting_all_edge_error['ds_name'], starting_all_edge_error['all_edge_errors']))
all_metrics_info['all_edge_errors_prop'] = all_metrics_info.apply(lambda row: row['all_edge_errors'] / starting_all_edge_error[row['ds_name']], axis=1)

In [ ]:
# compute the number of corrected edges for each introduced vertex
all_metrics_info["introduced"] = (
    all_metrics_info.groupby("ds_name")["fn_nodes"]
      .diff()
      .abs()
      .fillna(0)
)
all_metrics_info['corrected_edges'] = (
    all_metrics_info.groupby("ds_name")["all_edge_errors"]
      .diff()
      .abs()
      .fillna(0)
)
all_metrics_info['corrections_per_introduced'] = all_metrics_info.corrected_edges / all_metrics_info.introduced

In [12]:
# compute the number of corrected edges for each introduced vertex by comparing
# starting FN nodes and ending FN nodes, FP edges and FN edges
ds_names_with_merges_resolved = [
    ds_name for ds_name in os.listdir(ending_solution_root) if os.path.isdir(os.path.join(ending_solution_root, ds_name))
]
start_fn_nodes = []
end_fn_nodes = []
start_fp_edges = []
end_fp_edges = []
start_fn_edges = []
end_fn_edges = []
for ds_name in ds_names_with_merges_resolved:
    in_dir_path = os.path.join(
        ds_name,
        'matched_solution.zarr',
        'traccuracy-results.json'
    )
    starting_metrics_path = os.path.join(
        starting_solution_root,
        in_dir_path
    )
    ending_metrics_path = os.path.join(
        ending_solution_root,
        in_dir_path
    )
    for i, pth in enumerate([starting_metrics_path, ending_metrics_path]):
        with open(pth) as f:
            results = json.load(f)['traccuracy'][0]['results']
            fn_nodes = results['False Negative Nodes']
            fp_edges = results['False Positive Edges']
            fn_edges = results['False Negative Edges']
            if i == 0:
                start_fn_nodes.append(fn_nodes)
                start_fp_edges.append(fp_edges)
                start_fn_edges.append(fn_edges)
            else:
                end_fn_nodes.append(fn_nodes)
                end_fp_edges.append(fp_edges)
                end_fn_edges.append(fn_edges)
error_diff_df = pd.DataFrame(
    {
        'ds_name': ds_names_with_merges_resolved,
        'start_fn_nodes': start_fn_nodes,
        'end_fn_nodes': end_fn_nodes,
        'start_fp_edges': start_fp_edges,
        'end_fp_edges': end_fp_edges,
        'start_fn_edges': start_fn_edges,
        'end_fn_edges': end_fn_edges
    }
)
error_diff_df['introduced_fn_nodes'] = error_diff_df['start_fn_nodes'] - error_diff_df['end_fn_nodes']
error_diff_df['corrected_fp_edges'] = error_diff_df['start_fp_edges'] - error_diff_df['end_fp_edges']
error_diff_df['corrected_fn_edges'] = error_diff_df['start_fn_edges'] - error_diff_df['end_fn_edges']
error_diff_df['total_corrected_edges'] = error_diff_df['corrected_fp_edges'] + error_diff_df['corrected_fn_edges']
error_diff_df['corrected_per_introduced'] = error_diff_df['total_corrected_edges'] / error_diff_df['introduced_fn_nodes']
error_diff_df[['ds_name', 'introduced_fn_nodes', 'total_corrected_edges', 'corrected_per_introduced']].sort_values(by='ds_name')

,ds_name,introduced_fn_nodes,total_corrected_edges,corrected_per_introduced
2,BF-C2DL-HSC_01,1,4,4.000000
4,BF-C2DL-HSC_02,2,6,3.000000
9,BF-C2DL-MuSC_01,15,45,3.000000
1,BF-C2DL-MuSC_02,59,202,3.423729
11,Fluo-C3DL-MDA231_01,1,2,2.000000
10,Fluo-N2DH-GOWT1_01,1,4,4.000000
5,Fluo-N2DH-GOWT1_02,12,16,1.333333
12,Fluo-N2DL-HeLa_01,24,42,1.750000
6,Fluo-N2DL-HeLa_02,31,63,2.032258
3,Fluo-N3DH-CE_01,14,47,3.357143


In [ ]:
# define function to only keep the iterations for a dataset where a value changed
def truncate_when_constant(df, group_col, value_col, x_col="iteration"):
    """Keep rows up to the last change in `value_col` for each group in `group_col`."""
    truncated = []
    for name, sub in df.groupby(group_col):
        sub = sub.sort_values(x_col)
        # Find where value stops changing
        last_change_idx = sub[value_col].ne(sub[value_col].shift()).cumsum().idxmax()
        truncated.append(sub.loc[:last_change_idx])
    return pd.concat(truncated)

In [ ]:
y_vars = ['fn_edges_prop', 'fp_edges_prop', 'all_edge_errors_prop']
truncated_dfs = {}
for y in y_vars:
    truncated_dfs[y] = truncate_when_constant(all_metrics_info, "ds_name", y)